## Setup

This section prepares the environment and project path.

In [16]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("src exists:", (project_root / "src").exists())

Project root: c:\Users\harsh\facial-emotion-recognition-cv
src exists: True


## Imports

This section loads required libraries for real-time detection.

In [17]:
from pathlib import Path

import torch
import numpy as np
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.deep_learning.custom_cnn import CustomCNN

## Load model

This section loads the trained CNN model.

In [21]:
class_names = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = Path("../outputs/checkpoints/custom_cnn_clean_run.pth")

print("Model path:", model_path.resolve())
print("Exists:", model_path.exists())

model = CustomCNN(num_classes=len(class_names))
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval()

print("Model loaded successfully")

Model path: C:\Users\harsh\facial-emotion-recognition-cv\outputs\checkpoints\custom_cnn_clean_run.pth
Exists: True


C:\Users\harsh\AppData\Local\Temp\ipykernel_24856\177126346.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=de

Model loaded successfully


## Preprocessing

This section defines the image transformation pipeline.

In [22]:
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

test_dataset = datasets.ImageFolder(root="../data/raw/fer2013/test", transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

test_accuracy = accuracy_score(all_labels, all_preds)
print(f"Checkpoint Test Accuracy: {test_accuracy:.4f}")

Checkpoint Test Accuracy: 0.5763


## Face detection

This section loads the Haar cascade model.

In [23]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

print("Face detector loaded")

Face detector loaded


## Real-time detection

This section captures webcam frames and predicts emotions.

In [ ]:
import cv2
import torch
from pathlib import Path
from collections import deque, Counter
from torchvision import transforms

from src.deep_learning.custom_cnn import CustomCNN

class_names = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = Path("../outputs/checkpoints/custom_cnn_aug_best.pth")
print("Model path:", model_path.resolve())
print("Exists:", model_path.exists())

model = CustomCNN(num_classes=len(class_names))
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

prediction_buffer = deque(maxlen=12)
stable_label = "uncertain"
stable_conf = 0.0
frame_count = 0
update_every = 4

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam")
else:
    print("Press 'q' to exit")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=6,
        minSize=(80, 80)
    )

    if len(faces) > 0:
        faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
        x, y, w, h = faces[0]

        pad = int(0.08 * w)
        x1 = max(x - pad, 0)
        y1 = max(y - pad, 0)
        x2 = min(x + w + pad, gray.shape[1])
        y2 = min(y + h + pad, gray.shape[0])

        face = gray[y1:y2, x1:x2]

        h_face, w_face = face.shape
        cx1 = int(0.10 * w_face)
        cy1 = int(0.10 * h_face)
        cx2 = int(0.90 * w_face)
        cy2 = int(0.90 * h_face)

        face_center = face[cy1:cy2, cx1:cx2]
        face_resized = cv2.resize(face_center, (48, 48))

        if frame_count % update_every == 0:
            input_tensor = transform(face_resized).unsqueeze(0).to(device)

            with torch.no_grad():
                output = model(input_tensor)
                probs = torch.softmax(output, dim=1).squeeze(0)

            top2_probs, top2_idx = torch.topk(probs, 2)
            top1_conf = top2_probs[0].item()
            top2_conf = top2_probs[1].item()
            pred_idx = top2_idx[0].item()
            pred_label = class_names[pred_idx]

            margin = top1_conf - top2_conf

            if top1_conf >= 0.35 and margin >= 0.08:
                prediction_buffer.append(pred_label)
            else:
                prediction_buffer.append("uncertain")

            most_common = Counter(prediction_buffer).most_common(1)[0][0]

            if most_common != "uncertain":
                stable_label = most_common
                stable_conf = top1_conf

        display_label = f"{stable_label} ({stable_conf:.2f})"

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            display_label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Model path: C:\Users\harsh\facial-emotion-recognition-cv\outputs\checkpoints\custom_cnn_aug_best.pth
Exists: True


C:\Users\harsh\AppData\Local\Temp\ipykernel_24856\3682005423.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=d

Press 'q' to exit


KeyboardInterrupt: 

: 